<a href="https://colab.research.google.com/github/rem27nean/kaybee-27/blob/main/Rem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
data = [
  {"cv": "Python developer with machine learning experience",
   "job": "Looking for a data scientist with Python and ML",
   "label": 1},

  {"cv": "Graphic designer skilled in Photoshop and Illustrator",
   "job": "Backend developer needed with Java and Spring Boot",
   "label": 0},

  {"cv": "Data analyst with SQL, Excel, and Power BI",
   "job": "Hiring data analyst with SQL and visualization skills",
   "label": 1},

  {"cv": "Civil engineer with construction experience",
   "job": "Software engineer required with Python skills",
   "label": 0}
]

In [ ]:
from datasets import Dataset


dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size = 0.2)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['cv', 'job', 'label'],
        num_rows: 3
    })
    test: Dataset({
        features: ['cv', 'job', 'label'],
        num_rows: 1
    })
})


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(example):
    return tokenizer(
        example['cv'],
        example['job'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

dataset = dataset.map(tokenize)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir='./logs'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=0.8652993440628052, metrics={'train_runtime': 36.6482, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.109, 'total_flos': 394666583040.0, 'train_loss': 0.8652993440628052, 'epoch': 2.0})

In [ ]:
import torch

def predict(cv,job):
  inputs = tokenizer(cv, job, return_tensors = "pt", truncation = True, padding = True)
  outputs = model(**inputs)
  probs = torch.nn.functional.softmax(outputs.logits, dim = 1)
  return probs [0][1].item()

print(predict(
    "Python developer with data science skills",
    "Looking for machine learning engineer with Python"
    ))

0.4286903142929077


In [ ]:
model.save_pretrained("my_model")
tokenizer.save_pretrained("my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_model/tokenizer_config.json', 'my_model/tokenizer.json')

In [ ]:
!zip -r my_model.zip my_model

  adding: my_model/ (stored 0%)
  adding: my_model/config.json (deflated 53%)
  adding: my_model/model.safetensors (deflated 7%)
  adding: my_model/tokenizer.json (deflated 71%)
  adding: my_model/tokenizer_config.json (deflated 42%)


In [ ]:
from google.colab import files
files.download("my_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>